# 13 - Exploratory data analysis
Statistics and charts that inform cleaning, transformation, feature engineering, splitting and evaluation.
Uses only `tennis_features.model_dev_view` (warmup, train and dev: seasons 2010-2024). Test (2025) and holdout (2026) rows are not loaded, so no decision taken here has seen the evaluation data.
Read-only: figures are saved to `figures/`, downloaded as a zip and copied to the bucket under `reports/eda/`.
Run after 10_split and 08_weather_raw.

In [ ]:
CODE_VERSION = "code-v2"      # the git tag of the code being run: create it on GitHub before running

## Install pinned libraries
Read directly from the Github repo

In [ ]:
# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Authenticate and setup
Enable Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "13_eda"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers
matplotlib is not yet in requirements.txt: the version printed below should be added there (and a new code tag made).

In [ ]:
import gzip, json, os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
print("matplotlib", matplotlib.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
GREY, ACCENT = "#8c8c8c", "#2a6fdb"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9, "axes.titlesize": 10,
                     "figure.constrained_layout.use": True})       # keeps titles and labels from overlapping

def save(fig, name):
    """Save a figure for the report and show it."""
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def rate_ci(s):
    """Share of 1s in a 0/1 series, with a 95% interval (normal approximation)."""
    s = s.dropna()
    n, p = len(s), s.mean()
    h = 1.96 * np.sqrt(p * (1 - p) / n) if n else np.nan
    return pd.Series({"n": n, "rate": p, "lo": p - h, "hi": p + h})

def binned(d, x, y, q=10):
    """Mean of y (with 95% interval) in q quantile bins of x."""
    d = d[[x, y]].dropna()
    d = d.assign(bin=pd.qcut(d[x], q, duplicates="drop"))
    out = d.groupby("bin", observed=True)[y].apply(rate_ci).unstack()
    out["x"] = d.groupby("bin", observed=True)[x].median()
    return out

def auc(score, label):
    """Area under the ROC curve of one feature (Mann-Whitney), ignoring missing values."""
    d = pd.DataFrame({"s": score, "y": label}).dropna()
    n1, n0 = (d["y"] == 1).sum(), (d["y"] == 0).sum()
    r = d["s"].rank()
    return (r[d["y"] == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def ci_plot(ax, t, labels=None, ref=None, horizontal=True):
    """Points with 95% intervals for a rate_ci table; ref = a reference line."""
    pos = np.arange(len(t))
    lab = labels if labels is not None else [str(i) for i in t.index]
    err = [t["rate"] - t["lo"], t["hi"] - t["rate"]]
    if horizontal:
        ax.errorbar(t["rate"], pos, xerr=err, fmt="o", color=ACCENT, ecolor=GREY, capsize=2)
        ax.set_yticks(pos, lab)
        if ref is not None: ax.axvline(ref, color=GREY, ls="--", lw=1)
    else:
        ax.errorbar(pos, t["rate"], yerr=err, fmt="o-", color=ACCENT, ecolor=GREY, capsize=2)
        ax.set_xticks(pos, lab, rotation=45)
        if ref is not None: ax.axhline(ref, color=GREY, ls="--", lw=1)

## 1. Load the development data
One query loads the view (about 34,000 rows); everything after this runs in pandas. Walkovers are kept in `df` (for counts) but left out of `m`, the modelling rows.

In [ ]:
VIEW = f"{PROJECT_ID}.tennis_features.model_dev_view"
EDITIONS = f"{PROJECT_ID}.tennis_clean.tournament_editions"

df = run(f"SELECT * FROM `{VIEW}`")
assert set(df["split"]) == {"warmup", "train", "dev"}, "The view must not contain test or holdout rows"

# BigQuery integers arrive as nullable Int64; plain floats are simpler for analysis
for c in df.columns:
    if str(df[c].dtype) == "Int64":
        df[c] = df[c].astype("float64")
for c in ["tourney_date", "match_date_min", "match_date_max"]:
    df[c] = pd.to_datetime(df[c])
df["modelling_ok"] = df["modelling_ok"].astype(bool)

m = df[df["modelling_ok"]].copy()

# Pre-match quantities used throughout (A is better ranked when log_rank_ratio > 0)
ranked = m["a_rank"].notna() & m["b_rank"].notna() & (m["a_rank"] != m["b_rank"])
m["log_rank_ratio"] = np.log(m["b_rank"]) - np.log(m["a_rank"])
m["rank_diff"] = m["b_rank"] - m["a_rank"]
m["fav_won"] = np.where(ranked, np.where(m["a_rank"] < m["b_rank"], m["a_won"], 1 - m["a_won"]), np.nan)

print("rows:", len(df), "| modelling rows:", len(m), "| walkovers:", int((~df["modelling_ok"]).sum()))
print(m["split"].value_counts().reindex(["warmup", "train", "dev"]))
print("seasons:", int(m["season"].min()), "-", int(m["season"].max()))

### Player-level view
Each match as two rows, one per player (`won` = 1 for the winner). Used for player distributions and history features.

In [ ]:
FIELDS = ["id", "name", "seed", "entry", "hand", "ht", "ioc", "age", "rank", "rank_points"]
STATS = ["ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "SvGms", "bpSaved", "bpFaced"]
CTX = ["match_id", "tourney_id", "tourney_date", "season", "split", "tourney_level", "surface", "round", "post_minutes", "post_is_retirement"]

def side(d, me, opp, won):
    out = d[CTX].copy()
    for f in FIELDS:
        out[f] = d[f"{me}_{f}"]
        out[f"opp_{f}"] = d[f"{opp}_{f}"]
    for s in STATS:
        out[s] = d[f"post_{me}_{s}"]
    out["won"] = won
    return out

long = pd.concat([side(m, "a", "b", m["a_won"]), side(m, "b", "a", 1 - m["a_won"])], ignore_index=True)
print(len(long), "player-match rows |", long["id"].nunique(), "players")

## 2. Target and the ranking baseline
### 2a. How often does the better-ranked player win?
**Question:** How strong is ranking alone, and does it differ by split, level, surface and round?

**Informs:** evaluation (the baseline to beat, per split); feature engineering (interactions with level, surface, round).

**Interpretation:** _to write_

In [ ]:
fav = m[m["fav_won"].notna()]
overall = fav["fav_won"].mean()
for col in ["split", "tourney_level", "surface", "round"]:
    print(fav.groupby(col)["fav_won"].apply(rate_ci).unstack().round(3), "\n")

ROUNDS = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharex=True)
for ax, col, order in [(axes[0], "tourney_level", ["G", "PM", "P", "I"]),
                       (axes[1], "surface", ["hard", "clay", "grass", "carpet"]),
                       (axes[2], "round", ROUNDS)]:
    t = fav.groupby(col)["fav_won"].apply(rate_ci).unstack().reindex([o for o in order if o in fav[col].unique()])
    ci_plot(ax, t, ref=overall)
    ax.set_title(f"by {col}")
    ax.invert_yaxis()
axes[1].set_xlabel("share of matches won by the better-ranked player (95% CI)")
fig.suptitle(f"Share of matches won by the better-ranked player, by level, surface and round (dashed line: all matches, {overall:.1%})", x=0.01, ha="left")
save(fig, "2a_favourite_win_rate_by_context")

### 2b. Shape of the ranking signal
**Question:** Is the relation between ranking gap and win probability linear in the rank difference or in the log rank ratio?

**Informs:** transformation (log of ranks), model choice (a near-linear signal on the logit scale suits logistic regression).

**Interpretation:** _to write_

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, x, label in [(axes[0], "rank_diff", "rank difference (B rank - A rank)"),
                     (axes[1], "log_rank_ratio", "log rank ratio  log(B rank / A rank)")]:
    t = binned(m, x, "a_won", q=20)
    ax.errorbar(t["x"], t["rate"], yerr=[t["rate"] - t["lo"], t["hi"] - t["rate"]], fmt="o", color=ACCENT, ecolor=GREY, capsize=2)
    ax.axhline(0.5, color=GREY, ls="--", lw=1)
    ax.set_xlabel(label + ", median of 20 equal-size bins")
axes[0].set_ylabel("share won by A (95% CI)")
fig.suptitle("Share won by player A against the ranking gap, on two scales", x=0.01, ha="left")
save(fig, "2b_ranking_signal_shape")

print("AUC of rank difference:", round(auc(m["rank_diff"], m["a_won"]), 4),
      "| AUC of log rank ratio:", round(auc(m["log_rank_ratio"], m["a_won"]), 4))

### 2c. Ranking points: scale over time
**Question:** Are ranking points comparable across seasons, and how closely do they follow rank?

**Informs:** transformation (points may need scaling within season, or rank used instead); multicollinearity between rank and points.

**Interpretation:** _to write_

In [ ]:
print("Spearman correlation of rank and points, per season:")
print(long.groupby("season")[["rank", "rank_points"]].corr(method="spearman").xs("rank", level=1)["rank_points"].round(3).to_string())

bands = pd.cut(long["rank"], [0, 10, 50, 100, 2000], labels=["1-10", "11-50", "51-100", "101+"])
pts = long.assign(band=bands).groupby(["season", "band"], observed=True)["rank_points"].median().unstack()
fig, ax = plt.subplots(figsize=(8, 3.4))
for i, b in enumerate(pts.columns):
    ax.plot(pts.index, pts[b], marker="o", ms=3, color=ACCENT, alpha=1 - 0.22 * i, label=f"rank {b}")
ax.set_yscale("log"); ax.set_ylabel("median ranking points (log scale)"); ax.set_xlabel("season")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
ax.set_title("Median ranking points per rank band and season")
save(fig, "2c_rank_points_by_season")

## 3. Distributions of pre-match player attributes
**Question:** What do rank, age, height, hand and entry type look like, and are there implausible values?

**Informs:** cleaning (outliers), transformation (log rank, capping), feature engineering (categorical encodings).

**Interpretation:** _to write_

In [ ]:
print(long[["rank", "rank_points", "age", "ht", "seed"]].describe(percentiles=[.01, .25, .5, .75, .99]).round(1))
print("\nhand:", long["hand"].value_counts(dropna=False).to_dict())
print("entry:", long["entry"].value_counts(dropna=False).to_dict())

print("\nPlausibility checks (player-match rows):")
print(" age under 15 or over 42:", int(((long["age"] < 15) | (long["age"] > 42)).sum()))
print(" height under 150 cm or over 200 cm:", int(((long["ht"] < 150) | (long["ht"] > 200)).sum()))
print(long.loc[(long["ht"] < 150) | (long["ht"] > 200), ["name", "ht"]].drop_duplicates().head(10))

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].hist(np.log10(long["rank"].dropna()), bins=40, color=ACCENT); axes[0].set_xlabel("log10(rank)"); axes[0].set_title("rank (log scale)")
axes[1].hist(long["age"].dropna(), bins=40, color=ACCENT); axes[1].set_xlabel("years"); axes[1].set_title("age")
axes[2].hist(long["ht"].dropna(), bins=30, color=ACCENT); axes[2].set_xlabel("cm"); axes[2].set_title(f"height ({long['ht'].isna().mean():.0%} missing)")
e = long["entry"].fillna("direct").value_counts()
axes[3].barh(e.index, e.values, color=[ACCENT if i == "direct" else GREY for i in e.index]); axes[3].invert_yaxis(); axes[3].set_title("entry type"); axes[3].set_xscale("log")
fig.suptitle("Player attributes per player-match, 2010-2024", x=0.01, ha="left")
save(fig, "3_player_attribute_distributions")

## 4. Missingness over time, and whether it is random
**Question:** Which fields are missing, when, and is missingness related to player strength?

**Informs:** cleaning (imputation vs indicator), feature selection (drop fields missing in recent seasons), splitting (missingness changes over time).

**Interpretation:** _to write_

In [ ]:
miss = m.groupby("season")[["a_ht", "b_ht", "post_a_svpt", "post_b_svpt", "post_minutes", "a_rank", "b_rank"]].apply(lambda d: pd.Series({
    "height (either player)": (d["a_ht"].isna() | d["b_ht"].isna()).mean(),
    "serve stats": (d["post_a_svpt"].isna() | d["post_b_svpt"].isna()).mean(),
    "match minutes": d["post_minutes"].isna().mean(),
    "rank (either player)": (d["a_rank"].isna() | d["b_rank"].isna()).mean()}))
print((100 * miss).round(1))

fig, ax = plt.subplots(figsize=(8, 3.4))
for i, c in enumerate(miss.columns):
    ax.plot(miss.index, 100 * miss[c], marker="o", ms=3, label=c, color=ACCENT if c.startswith("height") else GREY,
            ls=["-", "--", ":", "-."][i])
ax.axvspan(2009.5, 2015.5, color=GREY, alpha=0.08); ax.text(2010, 92, "warmup (history only)", fontsize=8, color=GREY)
ax.set_ylabel("% of modelling rows missing"); ax.set_xlabel("season"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Missing values per season (modelling rows)")
save(fig, "4_missingness_by_season")

In [ ]:
# Is missing height related to strength? Compare ranks of players with and without a height
h = long.assign(ht_missing=long["ht"].isna())
print(h.groupby(["split", "ht_missing"])["rank"].median().unstack().rename(columns={False: "median rank, height known", True: "median rank, height missing"}))
print("\nWin rate of players with missing height:", h.groupby("ht_missing")["won"].mean().round(3).to_dict())

## 5. Validity of the match statistics
**Question:** Are the post-match serve statistics and durations internally consistent?

**Informs:** cleaning before any history features are built from them (rows to fix, flag or drop).

**Interpretation:** _to write_

In [ ]:
s = long.dropna(subset=["svpt"])
rules = {
    "first serves in > serve points": s["1stIn"] > s["svpt"],
    "first-serve points won > first serves in": s["1stWon"] > s["1stIn"],
    "second-serve points won > second serves": s["2ndWon"] > (s["svpt"] - s["1stIn"]),
    "double faults > second serves": s["df"] > (s["svpt"] - s["1stIn"]),
    "break points saved > faced": s["bpSaved"] > s["bpFaced"],
    "zero serve points": s["svpt"] == 0,
}
print(pd.Series({k: int(v.sum()) for k, v in rules.items()}, name="player-match rows").to_string())

mins = m.dropna(subset=["post_minutes"])
print("\nminutes:", mins["post_minutes"].describe(percentiles=[.01, .5, .99]).round(0).to_dict())
print("matches under 30 minutes:", int((mins["post_minutes"] < 30).sum()),
      "| of which retirements:", int(((mins["post_minutes"] < 30) & mins["post_is_retirement"]).sum()))
print("matches over 240 minutes:", int((mins["post_minutes"] > 240).sum()))
print(mins.loc[mins["post_minutes"] > 240, ["season", "tourney_name", "round", "post_score", "post_minutes"]].head(10))

## 6. Context: surface mix, upsets by entry type
**Question:** How does the surface mix change over time, and do qualifiers, wildcards and lucky losers win more or less than their rank suggests?

**Informs:** feature engineering (surface-specific ratings, entry flags), cleaning (carpet category).

**Interpretation:** _to write_

In [ ]:
mix = m.groupby(["season", "surface"]).size().unstack(fill_value=0)
mix = mix.div(mix.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2))
bottom = np.zeros(len(mix))
for i, sfc in enumerate([c for c in ["hard", "clay", "grass", "carpet"] if c in mix.columns]):
    ax.bar(mix.index, mix[sfc], bottom=bottom, color=ACCENT, alpha=1 - 0.25 * i, label=sfc)
    bottom += mix[sfc].values
ax.set_ylabel("share of matches"); ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1)); ax.set_title("Surface mix per season")
save(fig, "6a_surface_mix_by_season")

# Entry type: actual win rate vs the rate expected from the ranking gap alone (binned log rank ratio)
lr = long.assign(lrr=np.log(long["opp_rank"]) - np.log(long["rank"]))
lr["bin"] = pd.qcut(lr["lrr"], 20, duplicates="drop")
lr["expected"] = lr.groupby("bin", observed=True)["won"].transform("mean")
ent = lr.dropna(subset=["lrr"]).assign(entry=lambda d: d["entry"].fillna("direct")).groupby("entry").agg(
    n=("won", "size"), actual=("won", "mean"), expected_from_rank=("expected", "mean"))
ent["actual_minus_expected"] = ent["actual"] - ent["expected_from_rank"]
print(ent.sort_values("n", ascending=False).round(3))

## 7. Player history and cold start
**Question:** How many earlier matches do players have at match time, and does the ranking baseline weaken when one player has little history?

**Informs:** splitting (justifies the 2010-2015 warmup used only as history), feature engineering (shrinkage or default values for new players).

**Interpretation:** _to write_

In [ ]:
# Matches each player played in tournaments that started before this one (same-tournament rounds not counted)
per_day = long.groupby(["id", "tourney_date"]).size().rename("n").reset_index().sort_values(["id", "tourney_date"])
per_day["prior"] = per_day.groupby("id")["n"].cumsum() - per_day["n"]
long = long.drop(columns=["prior"], errors="ignore").merge(per_day[["id", "tourney_date", "prior"]], on=["id", "tourney_date"], how="left")

# The same count if history only started in 2016 (no warmup)
pd16 = per_day[per_day["tourney_date"] >= "2016-01-01"].copy()
pd16["prior_no_warmup"] = pd16.groupby("id")["n"].cumsum() - pd16["n"]
long = long.drop(columns=["prior_no_warmup"], errors="ignore").merge(pd16[["id", "tourney_date", "prior_no_warmup"]], on=["id", "tourney_date"], how="left")

t = long[long["split"] != "warmup"].groupby("split").agg(
    pct_under_10_prior=("prior", lambda s: 100 * (s < 10).mean()),
    pct_under_10_prior_without_warmup=("prior_no_warmup", lambda s: 100 * (s < 10).mean()),
    median_prior=("prior", "median"))
print(t.round(1))

for side_, col in [("a", "a_prior"), ("b", "b_prior")]:
    m = m.drop(columns=[col], errors="ignore").merge(
        long[["match_id", "id", "prior"]].rename(columns={"id": f"{side_}_id", "prior": col}), on=["match_id", f"{side_}_id"], how="left")
m["min_prior"] = m[["a_prior", "b_prior"]].min(axis=1)

dev_train = m[m["split"] != "warmup"]
g = dev_train.groupby(pd.cut(dev_train["min_prior"], [-1, 4, 9, 24, 49, 99, 10_000],
                             labels=["0-4", "5-9", "10-24", "25-49", "50-99", "100+"]), observed=True)["fav_won"].apply(rate_ci).unstack()
print(g.round(3))
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ci_plot(ax, g, horizontal=False, ref=dev_train["fav_won"].mean())
ax.set_xlabel("earlier matches of the less experienced player"); ax.set_ylabel("better-ranked player wins")
ax.set_title("Better-ranked player's win rate by the history of the less experienced player (train + dev)")
save(fig, "7_cold_start")

## 8. Recent form beyond ranking
**Question:** Does a player's win rate over the previous 52 weeks carry information that the ranking does not?

**Informs:** feature engineering (rolling form features built only from earlier tournaments, so no leakage).

**Interpretation:** _to write_

In [ ]:
# Wins and matches in the 365 days before each tournament start (earlier tournaments only: closed='left')
daily = long.groupby(["id", "tourney_date"]).agg(w=("won", "sum"), n=("won", "size")).reset_index().sort_values(["id", "tourney_date"])
roll = (daily.set_index("tourney_date").groupby("id")[["w", "n"]]
             .rolling("365D", closed="left").sum().reset_index())
roll["form"] = np.where(roll["n"] >= 5, roll["w"] / roll["n"], np.nan)      # at least 5 matches, else unknown
for side_ in ["a", "b"]:
    m = m.drop(columns=[f"{side_}_form"], errors="ignore").merge(
        roll[["id", "tourney_date", "form"]].rename(columns={"id": f"{side_}_id", "form": f"{side_}_form"}),
        on=[f"{side_}_id", "tourney_date"], how="left")
m["form_diff"] = m["a_form"] - m["b_form"]

dt = m[m["split"] != "warmup"]
close = dt[dt["log_rank_ratio"].abs() < 0.3]          # players of similar rank
print("rows with form for both players:", f"{dt['form_diff'].notna().mean():.1%}")
print("AUC, all train+dev rows: log rank ratio", round(auc(dt["log_rank_ratio"], dt["a_won"]), 4),
      "| form diff", round(auc(dt["form_diff"], dt["a_won"]), 4))
print("AUC, similar-rank matches only: log rank ratio", round(auc(close["log_rank_ratio"], close["a_won"]), 4),
      "| form diff", round(auc(close["form_diff"], close["a_won"]), 4), "| n =", len(close))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, d, title in [(axes[0], dt, "all train + dev matches"), (axes[1], close, "players of similar rank (|log rank ratio| < 0.3)")]:
    t = binned(d, "form_diff", "a_won", q=10)
    ax.errorbar(t["x"], t["rate"], yerr=[t["rate"] - t["lo"], t["hi"] - t["rate"]], fmt="o", color=ACCENT, ecolor=GREY, capsize=2)
    ax.axhline(0.5, color=GREY, ls="--", lw=1); ax.set_title(title); ax.set_xlabel("form difference: A minus B, 52-week win rate")
axes[0].set_ylabel("share won by A (95% CI)")
fig.suptitle("Share won by player A against the difference in 52-week form", x=0.01, ha="left")
save(fig, "8_recent_form")

## 9. Weather
**Question:** What weather do matches see, how are the weather variables related, and is there any sign that conditions change how often the favourite wins?

**Informs:** feature engineering (which weather variables to keep, aggregation over the match-day window), known limitations (indoor events are not flagged).

**Interpretation:** _to write_

In [ ]:
# Hourly weather from the packed file named in the latest weather manifest
names = sorted(b.name for b in bucket.list_blobs(prefix="manifests/open_meteo_archive_v2_"))
WEATHER_FILE = json.loads(bucket.blob(names[-1]).download_as_text())["files"][0]["file"]
assert WEATHER_FILE.endswith(".jsonl.gz"), "Run 08_weather_raw_revised first (packed weather file)"

parts = []
for line in gzip.decompress(bucket.blob(WEATHER_FILE).download_as_bytes()).splitlines():
    rec = json.loads(line)
    h = pd.DataFrame(rec["response"]["hourly"])
    h["tourney_id"] = rec["tourney_id"]
    parts.append(h)
hourly = pd.concat(parts, ignore_index=True)
hourly["time"] = pd.to_datetime(hourly["time"])            # local time of the venue (timezone=auto)

# Daytime (10:00-20:00 local) summary per edition and day
day = (hourly[hourly["time"].dt.hour.between(10, 20)]
       .assign(date=lambda d: d["time"].dt.normalize())
       .groupby(["tourney_id", "date"])
       .agg(temp=("temperature_2m", "mean"), dew_point=("dew_point_2m", "mean"), humidity=("relative_humidity_2m", "mean"),
            wind=("wind_speed_10m", "mean"), gusts=("wind_gusts_10m", "max"), rain_mm=("precipitation", "sum"),
            cloud=("cloud_cover", "mean"), pressure=("surface_pressure", "mean"))
       .reset_index())

# Each match: average over its window of possible match days
win = m[["match_id", "tourney_id", "match_date_min", "match_date_max"]].copy()
win["days"] = (win["match_date_max"] - win["match_date_min"]).dt.days + 1
win = win.loc[win.index.repeat(win["days"])]
win["date"] = win["match_date_min"] + pd.to_timedelta(win.groupby(level=0).cumcount(), unit="D")
WVARS = ["temp", "dew_point", "humidity", "wind", "gusts", "rain_mm", "cloud", "pressure"]
mw = win.merge(day, on=["tourney_id", "date"], how="left").groupby("match_id")[WVARS].mean()
m = m.drop(columns=WVARS, errors="ignore").merge(mw, left_on="match_id", right_index=True, how="left")

editions = run(f"SELECT tourney_id, place, country_code, elevation_m FROM `{EDITIONS}`")
m = m.drop(columns=["elevation_m", "place"], errors="ignore").merge(editions[["tourney_id", "elevation_m", "place"]], on="tourney_id", how="left")
print("hourly rows:", len(hourly), "| matches with weather:", f"{m['temp'].notna().mean():.1%}")
print(m[WVARS + ["elevation_m"]].describe(percentiles=[.01, .5, .99]).round(1))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, v, unit in zip(axes, ["temp", "humidity", "wind", "rain_mm"], ["°C", "%", "km/h", "mm per day, 10:00-20:00"]):
    ax.hist(m[v].dropna(), bins=40, color=ACCENT); ax.set_title(v); ax.set_xlabel(unit)
fig.suptitle("Weather during the possible match days (daytime averages)", x=0.01, ha="left")
save(fig, "9a_weather_distributions")

corr = m[WVARS + ["elevation_m"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=60); ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.grid(False); fig.colorbar(im, shrink=0.8); ax.set_title("Spearman correlation between weather variables")
save(fig, "9b_weather_correlation")

In [ ]:
dt = m[(m["split"] != "warmup") & m["fav_won"].notna()]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
panels = [("temp", "temperature quintile (°C)"), ("humidity", "humidity quintile (%)"), ("wind", "wind quintile (km/h)")]
for ax, (v, label) in zip(axes, panels):
    q = pd.qcut(dt[v], 5)
    t = dt.groupby(q, observed=True)["fav_won"].apply(rate_ci).unstack()
    ci_plot(ax, t, labels=[f"{iv.left:.0f}-{iv.right:.0f}" for iv in t.index], horizontal=False, ref=dt["fav_won"].mean())
    ax.set_xlabel(label)
t = dt.groupby(pd.cut(dt["elevation_m"], [-100, 250, 750, 3000], labels=["<250 m", "250-750 m", ">750 m"]), observed=True)["fav_won"].apply(rate_ci).unstack()
ci_plot(axes[3], t, horizontal=False, ref=dt["fav_won"].mean()); axes[3].set_xlabel("venue elevation")
axes[0].set_ylabel("better-ranked player wins")
fig.suptitle("Favourite win rate by conditions (train + dev); dashed line = overall", x=0.01, ha="left")
save(fig, "9c_weather_vs_favourite")
print("AUC of each weather variable for 'favourite wins' (0.5 = no signal):")
print(pd.Series({v: round(auc(dt[v], dt["fav_won"]), 3) for v in WVARS + ["elevation_m"]}).to_string())

## 10. Change over time
**Question:** Is the relationship between ranking and winning stable across seasons?

**Informs:** splitting (time-ordered split, not random), evaluation (compare models with the baseline within each split), training (whether to weight recent seasons).

**Interpretation:** _to write_

In [ ]:
t = m.groupby("season")["fav_won"].apply(rate_ci).unstack()
print(t.round(3))
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.errorbar(t.index, t["rate"], yerr=[t["rate"] - t["lo"], t["hi"] - t["rate"]], fmt="o-", color=ACCENT, ecolor=GREY, capsize=2)
for name, lo, hi in [("warmup", 2010, 2015), ("train", 2016, 2022), ("dev", 2023, 2024)]:
    ax.axvspan(lo - 0.5, hi + 0.5, color=GREY, alpha=0.06 if name != "train" else 0.12)
    ax.text((lo + hi) / 2, t["lo"].min() - 0.004, name, ha="center", va="top", fontsize=8, color=GREY)
ax.set_ylim(t["lo"].min() - 0.012, t["hi"].max() + 0.005)
ax.set_ylabel("better-ranked player wins"); ax.set_xlabel("season"); ax.set_title("Favourite win rate per season (95% CI)")
save(fig, "10_drift_favourite_win_rate")

## 11. Check: A/B assignment is symmetric
**Question:** Are players A and B alike, so the model cannot learn the side instead of the players?

**Informs:** validation of the reshape in notebook 09; no leakage through the A/B layout.

**Interpretation:** _to write_

In [ ]:
print("share won by A:", round(m["a_won"].mean(), 4))
print("A better ranked:", round((m["a_rank"] < m["b_rank"]).mean(), 4), "| B better ranked:", round((m["a_rank"] > m["b_rank"]).mean(), 4))
print(m[["a_rank", "b_rank", "a_age", "b_age", "a_ht", "b_ht"]].median().round(1).to_string())
print("mean log rank ratio (about 0 if symmetric):", round(m["log_rank_ratio"].mean(), 4))

## 12. Candidate features side by side
**Question:** Which candidate pre-match features carry signal on their own, and which overlap?

**Informs:** feature selection for the first model; multicollinearity (pairs with |rho| > 0.8).

**Interpretation:** _to write_

In [ ]:
m["log_points_ratio"] = np.log1p(m["a_rank_points"]) - np.log1p(m["b_rank_points"])
m["age_diff"] = m["a_age"] - m["b_age"]
m["ht_diff"] = m["a_ht"] - m["b_ht"]
m["prior_diff"] = np.log1p(m["a_prior"]) - np.log1p(m["b_prior"])
CANDIDATES = ["log_rank_ratio", "log_points_ratio", "form_diff", "age_diff", "ht_diff", "prior_diff"]

dt = m[m["split"] != "warmup"]
summary = pd.DataFrame({
    "auc_train_dev": {c: auc(dt[c], dt["a_won"]) for c in CANDIDATES},
    "pct_missing": {c: 100 * dt[c].isna().mean() for c in CANDIDATES}}).round(3)
print(summary.sort_values("auc_train_dev", ascending=False))

corr = dt[CANDIDATES].corr(method="spearman")
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=60); ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.grid(False); fig.colorbar(im, shrink=0.8); ax.set_title("Spearman correlation of candidate features (train + dev)")
save(fig, "12_candidate_feature_correlation")

## 13. Save the figures
Downloaded as a zip for the report, and copied to the bucket under `reports/eda/<code version>/`.

In [ ]:
import shutil
shutil.make_archive("eda_figures", "zip", FIG_DIR)
for f in sorted(os.listdir(FIG_DIR)):
    bucket.blob(f"reports/eda/{CODE_VERSION}/{f}").upload_from_filename(f"{FIG_DIR}/{f}", content_type="image/png")
print(len(os.listdir(FIG_DIR)), "figures saved to", f"gs://{BUCKET_NAME}/reports/eda/{CODE_VERSION}/")

from google.colab import files
files.download("eda_figures.zip")